In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agriculture-climate-slm-challenge/test_questions.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/sample_submission.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/train_qa.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/documents.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/dataset-metadata.json
/kaggle/input/competitions/agriculture-climate-slm-challenge/baseline_submission.csv


In [2]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/agriculture-climate-slm-challenge


In [3]:
base = "/kaggle/input/competitions/agriculture-climate-slm-challenge/"


In [4]:
import pandas as pd

train = pd.read_csv(base + "train_qa.csv")
test = pd.read_csv(base + "test_questions.csv")

train.head(), test.head()


(                                            question               topic  \
 0         Weevils in stored maize without chemicals?        post_harvest   
 1    Insurance paid but my field still failed — why?  climate_adaptation   
 2      Which cover crop helps between maize seasons?         soil_health   
 3  Maize stalks lodging before harvest — nutrient...          fertiliser   
 4                       What are signs of bean rust?       crop_diseases   
 
       crop  agro_zone  document_id  \
 0    maize  semi_arid  doc_pos_001   
 1  general  semi_arid  doc_cli_003   
 2  general  sub_humid  doc_soi_002   
 3    maize  sub_humid  doc_fer_001   
 4    beans   highland  doc_dis_002   
 
                                     reference_answer  QuestionId  
 0  Dry to twelve to thirteen percent and seal in ...           1  
 1  Basis risk means index payouts may not match i...           2  
 2  Mucuna or lablab reduce erosion and suppress w...           3  
 3  Ensure balanced NPK incl

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

rows = []

for _, t in test.iterrows():
    # Filter training questions by same topic
    sub = train[train["topic"] == t["topic"]]
    if sub.empty:
        sub = train

    vec = TfidfVectorizer(ngram_range=(1, 2))

    # FIRST fit on the training questions
    sub_matrix = vec.fit_transform(sub["question"])

    # THEN transform the test question
    test_vec = vec.transform([t["question"]])

    # Compute similarity
    sims = cosine_similarity(test_vec, sub_matrix)

    # Pick the best answer
    ans = sub.iloc[sims.argmax()]["reference_answer"]
    rows.append({"QuestionId": t["QuestionId"], "Answer": ans})

submission = pd.DataFrame(rows)
submission.to_csv("/kaggle/working/submission.csv", index=False)

submission


,QuestionId,Answer
0,1001,"Use clean seed, avoid dusk overhead irrigation..."
1,1002,"Respiratory distress, greenish diarrhoea, and ..."
2,1003,Likely nitrogen deficiency — yellowing starts ...
3,1004,Only if rain or irrigation is imminent; otherw...
4,1005,Ensure balanced NPK including potassium for st...
5,1006,Larvae feed in the whorl leaving ragged window...
6,1007,Acidity binds phosphorus and limits nodulation...
7,1008,Aphid honeydew leading to sooty mould; wash ap...
8,1009,"It should be dark, crumbly, and free of undeco..."
9,1010,Use short-season drought-tolerant varieties al...
